# Lesson 02 - Exploring Microsoft Agent Framework

The **Microsoft Agent Framework (MAF)** is a unified framework for building AI agents. It provides a clean, composable architecture with four core building blocks:

- **Client** – connects to an AI model endpoint and handles communication
- **Agent** – wraps a client with instructions and tool definitions
- **Tools** – extend agent capabilities with custom functions the model can call
- **Session** – maintains conversation history for multi-turn interactions

In this lesson, we'll build a **travel booking agent** that checks destination availability using these concepts.

## Setup

In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [1]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
from typing import Annotated

from dotenv import load_dotenv

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

load_dotenv()

<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


True

## Understanding the Agent Framework Architecture

The Microsoft Agent Framework follows a layered architecture:

```
Chat Client  →  Agent  →  Tools
                       →  Session
```

1. **Chat Client** – A `FoundryChatClient` connects to a model deployed in a Microsoft Foundry project. It handles authentication, request formatting, and response parsing. (If you target a raw Azure OpenAI / OpenAI endpoint instead, swap it for `OpenAIChatCompletionClient` from `agent_framework.openai`.)
2. **Agent** – Constructed directly via `Agent(client=..., ...)`, the agent combines model access with instructions (system prompt) and tools.
3. **Tools** – Python functions decorated with `@tool` that the agent can invoke to perform actions or retrieve data.
4. **Session** – An `AgentSession` object (created via `agent.create_session()`) that stores conversation history, enabling multi-turn dialogue where the agent remembers prior context.

Let's build each layer step by step.

**Environment variables expected:**

```
FOUNDRY_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
FOUNDRY_MODEL=gpt-4o-mini
```

Make sure you're signed in via `az login` so `AzureCliCredential` can authenticate.

In [2]:
# Create the chat client – this is the connection to the AI model.
client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

## Adding Tools with the @tool Decorator

Tools let agents take actions beyond generating text. The `@tool` decorator converts a regular Python function into something the agent can call.

Key points:
- Use `Annotated[type, "description"]` so the model understands each parameter.
- The docstring becomes the tool description the model sees.
- `approval_mode="never_require"` means the tool runs automatically without user confirmation. (This is the default; we set it explicitly here for clarity.) Use `"always_require"` instead when you want a human-in-the-loop approval gate before the tool fires.

In [9]:
# Shared catalog — in a real system this would come from a booking database.
AVAILABILITY = {
    "Barcelona": True,
    "Tokyo": True,
    "Cape Town": False,
    "Vancouver": True,
    "Dubai": False,
}


@tool(approval_mode="never_require")
def list_destinations() -> str:
    """List every destination the travel agency offers, regardless of availability."""
    return ", ".join(AVAILABILITY.keys())


@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a specific vacation destination is currently available for booking."""
    if destination not in AVAILABILITY:
        return f"{destination} is not in our catalog. Call list_destinations to see options."
    is_available = AVAILABILITY[destination]
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Creating an Agent with Tools

Now we combine the chat client, instructions, and tools into an agent. The `instructions` act as the system prompt — they define the agent's persona and behaviour.

In [10]:
agent = Agent(
    client=client,
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent for a small agency. "
        "You do not know the catalog from memory — always rely on the tools. "
        "When the user asks what destinations exist or which are available, "
        "call list_destinations first, then call check_destination_availability "
        "for each candidate before recommending one. "
        "Never invent destinations that the tools did not return."
    ),
    tools=[list_destinations, check_destination_availability],
)

## Multi-Turn Conversations with Sessions

By default, each `agent.run(...)` call is stateless — the model only sees the current message. An `AgentSession` (created via `agent.create_session()`) keeps track of all messages in a conversation. By passing the same session to each `agent.run()` call, the agent has access to the full conversation history and can refer back to earlier messages.

Tools registered at agent-creation time are available on every turn, so the agent can call our availability checker whenever the conversation needs it.

In [11]:
session = agent.create_session()

# Turn 1: Ask about available destinations — the agent should now call
# list_destinations and then check_destination_availability for each.
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

Agent: Currently available destinations are: Barcelona, Tokyo, and Vancouver.

Not currently available: Cape Town and Dubai.


In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

Agent: For somewhere warm, the available options are Barcelona and Tokyo.

Vancouver is also available, but I wouldn’t usually classify it as a warm destination. Cape Town and Dubai are not currently available for booking.


: 

## Summary

In this lesson you explored the four pillars of the Microsoft Agent Framework:

| Concept | What You Learned |
|---------|------------------|
| **Chat Client** | `FoundryChatClient` connects to a Microsoft Foundry project model with Azure credential-based auth (use `OpenAIChatCompletionClient` for raw Azure OpenAI / OpenAI endpoints) |
| **Agent** | `Agent(client=..., name=..., instructions=..., tools=[...])` bundles a model connection with a system prompt and capabilities |
| **Tools** | The `@tool` decorator exposes Python functions for the agent to call, with optional `approval_mode` for human-in-the-loop gating |
| **Session** | `agent.create_session()` maintains conversation history across multiple turns |

You also saw a common pitfall: an agent can only do what its tools allow. A `check_destination_availability(destination)` tool can verify a named destination, but it can't enumerate the catalog — that requires a separate `list_destinations()` tool. When the model's response is vague or evasive, it's usually a sign that the toolset doesn't cover the question being asked.

These building blocks compose together to create agents that can hold natural conversations, call external functions, and maintain context — the foundation for more advanced agentic patterns in later lessons (sequential pipelines with `SequentialBuilder`, parallel execution with `ConcurrentBuilder`, and full graph workflows with `WorkflowBuilder`).